# Preprocessing Technique: Normalization / Scaling (TF-IDF Vectors)
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Presented by:** Member 5 — *[Full Name, IT Number]*
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned preprocessing technique for our group's project,
as required for Progress Review I: technique explanation, justification, implementation, and
an interpreted EDA visualization.


## Shared Setup

This cell is identical across every member's notebook so each person's notebook can run
independently. It loads the assigned dataset and converts it to a pandas DataFrame.


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
train_df.head()


## 1. Technique Explanation

**Normalization/scaling** rescales numeric features onto a comparable range so that no single
feature dominates a model purely due to its scale. For text data, this applies to the
**TF-IDF vectors** produced from review text: each document's vector is normalized (commonly
to unit length, L2 normalization) so that longer documents don't automatically produce larger
raw feature values than shorter ones.


## 2. Justification for This Dataset

Reviews in our dataset vary in length (see Member 4's outlier analysis). Without normalization,
a longer review would naturally accumulate larger raw term-frequency values simply because it
contains more words — not because it is more sentiment-laden. L2 normalization removes this
length bias so the model compares documents based on their *word pattern*, not their *word
count*, which is especially important for a linear model like Logistic Regression.

## 3. Implementation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

sample_texts = train_df["text"].str.lower()

# Un-normalized TF-IDF (raw term-frequency weighted values)
vectorizer_raw = TfidfVectorizer(norm=None, max_features=5000)
X_raw = vectorizer_raw.fit_transform(sample_texts)

# L2-normalized TF-IDF (the standard, scaled version)
vectorizer_norm = TfidfVectorizer(norm="l2", max_features=5000)
X_norm = vectorizer_norm.fit_transform(sample_texts)

raw_vector_norms = np.sqrt(X_raw.multiply(X_raw).sum(axis=1)).A1
scaled_vector_norms = np.sqrt(X_norm.multiply(X_norm).sum(axis=1)).A1

print("Raw vector norm range:   ", raw_vector_norms.min(), "to", raw_vector_norms.max())
print("Scaled vector norm range:", scaled_vector_norms.min(), "to", scaled_vector_norms.max())


## 4. EDA Visualization & Interpretation

Histograms of document vector magnitude before and after normalization make the effect of
scaling directly visible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(raw_vector_norms, bins=40, ax=axes[0], color="#C1272D")
axes[0].set_title("Document Vector Magnitude\n(Before Normalization)")
axes[0].set_xlabel("Vector norm")

sns.histplot(scaled_vector_norms, bins=40, ax=axes[1], color="#2E9E6D")
axes[1].set_title("Document Vector Magnitude\n(After L2 Normalization)")
axes[1].set_xlabel("Vector norm")

plt.tight_layout()
plt.show()


**Interpretation:** [Fill in after running — expected result: the "before" histogram
shows a spread of vector magnitudes correlated with review length, while the "after" histogram
collapses to a single spike at 1.0 for every document, since L2 normalization forces every
vector onto the unit hypersphere. This confirms the scaling step removes length as a
confounding factor before the vectors are passed to a classifier.]
